# PE6201 A2 · Problem B · Shared Team Experiment Notebook

**Team B-8 — Outpatient Referral Coordination Agent**

This notebook is the team's shared **experiment, evaluation and evidence template**.

> **Implementation rule:** production logic belongs in the `.py` modules (`agent.py`, `tools.py`, `backends.py`, `guardrails.py`, `harness.py`, `prompt.py`).  
> **Notebook rule:** use this notebook to run experiments, inspect traces, aggregate results, create tables/plots, and write report-ready inferences.

## Project outcomes

The agent must end in exactly one of:

- `book`
- `request_information`
- `escalate`

The gated real-world action is **booking a clinic appointment**.

## Team target

- 50 evaluation cases total
- 15 supplied + 35 team-created
- scripted backend remains the reproducible default
- final live-model battery compares multiple model families on the same frozen system


## 0 · Team ownership map

| Member | Primary ownership | Live-model responsibility |
|---|---|---|
| **SYEDYASEEN ROSHAN TUSHAR** | D1, D2(a), D2(c) — loop, tools, parallel calls | GPT-4o-mini |
| **SHEN SHUO** | D1, D2(a), D2(c) — loop/tools support | Claude 3 Haiku |
| **GONG XINYI** | D2(b), D3 — descriptors + guardrails | Gemini 1.5 Flash |
| **LIU XINYAO** | D4, D5(a) — harness + scripted evaluation | Llama 3.1 8B |
| **XIE YULONG** | D6 — cost model, ledger, sensitivity | Mistral Nemo |
| **ZHONG YINGMEI** | report/demo assembly + v1 pass | GPT-4o-mini v1 pass |

### Everyone also does

1. Creates and labels their assigned evaluation cases.
2. Uses the same shared answer key (`expected_outcomes_B.json`).
3. Runs one assigned live-model pass (or the v1 pass).
4. Adds measured results to the shared result tables below.

### Important

Do **not** create separate evaluation logic per person.  
Do **not** create separate answer-key files per person.  
Do **not** copy module logic into notebook cells unless it is a short experiment override.


## 0.1 · GitHub working conventions

Recommended repository layout:

```text
PE6201-A2-Outpatient-Referral-Agent/
│
├── A2_scaffold/
│   ├── agent.py
│   ├── tools.py
│   ├── backends.py
│   ├── guardrails.py
│   ├── harness.py
│   ├── prompt.py
│   ├── config.py
│   ├── run_eval.py
│   └── demo_loop_failure.py
│
├── A2_reference_data/
│   ├── data_B/
│   ├── expected_outcomes_B.json
│   ├── make_fixtures_B.py
│   └── check_my_data.py
│
├── notebooks/
│   └── PE6201_A2_ProblemB_Team_Notebook.ipynb
│
├── results/
│   ├── scripted_final.json
│   ├── gpt4o_mini_v2.json
│   ├── claude_v2.json
│   └── ...
│
├── README.md
├── CONTRIBUTIONS.md
└── TEAM_DECLARATION.pdf
```

### Suggested Git workflow

- Each person works on a separate branch:
  - `roshan-loop-tools`
  - `shen-loop-support`
  - `gong-descriptors-guardrails`
  - `liu-eval-harness`
  - `xie-cost`
  - `zhong-report-demo`
- Keep commits small and descriptive.
- Avoid six people editing `backends.py` or this notebook at the same time.
- Merge module changes first, then refresh notebook outputs.


## 0.2 · Assignment-wide tracker

| Requirement | Owner | Status |
|---|---|---|
| D0 agent rationale | Team | ☐ |
| D1 working single-agent loop | Roshan / Shen | ☐ |
| D2(a) justified tool set | Roshan / Shen | ☐ |
| D2(b) v1→v2 descriptor experiment | Gong | ☐ |
| D2(c) sequential vs parallel | Roshan / Shen | ☐ |
| D3 ≥10 guardrail cases | Gong | ☐ |
| D4 50-case labelled eval set | All / Liu | ☐ |
| D4 code + judgement checks | Liu | ☐ |
| D5(a) scripted full run | Liu | ☐ |
| D5(b) live model battery | All | ☐ |
| Actual provider token usage captured | All live runners | ☐ |
| D6 cost-to-serve | Xie | ☐ |
| D6 ±10pp sensitivity | Xie | ☐ |
| D6 break-even | Xie | ☐ |
| D7 failure #1 | Team | ☐ |
| D7 failure #2 | Team | ☐ |
| Extended answer key committed | All / Liu | ☐ |
| Results committed | Liu / Team | ☐ |
| README reproduction steps | Zhong / Team | ☐ |
| CONTRIBUTIONS.md aligned | Team | ☐ |


# 1 · Shared setup

Run this first.

The notebook assumes the scaffold and reference-data folders live in the same repository.  
If your local paths differ, edit only `REPO_ROOT` / `SCAFFOLD_DIR` / `DATA_ROOT` below.


In [ ]:
from pathlib import Path
import sys, json, statistics, math
from pprint import pprint

# ---- EDIT ONLY IF YOUR LOCAL REPO LAYOUT DIFFERS ----
REPO_ROOT = Path.cwd()

# If notebook is under notebooks/, move to repo root.
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

SCAFFOLD_DIR = REPO_ROOT / "A2_scaffold"
DATA_ROOT = REPO_ROOT / "A2_reference_data"

if str(SCAFFOLD_DIR) not in sys.path:
    sys.path.insert(0, str(SCAFFOLD_DIR))

print("REPO_ROOT :", REPO_ROOT.resolve())
print("SCAFFOLD  :", SCAFFOLD_DIR.resolve())
print("DATA_ROOT :", DATA_ROOT.resolve())


In [ ]:
# Import shared implementation modules.
# If an import fails, fix the repository path/layout before continuing.

import config
import agent
import tools
import backends
import guardrails
import harness
import prompt

print("BACKEND :", getattr(config, "BACKEND", None))
print("PROBLEM :", getattr(config, "PROBLEM", None))
print("MODEL   :", getattr(config, "MODEL", None))
print("MAX_TURNS:", getattr(config, "MAX_TURNS", None))
print("AUTONOMY :", getattr(config, "AUTONOMY", None))


# 2 · Load and validate Problem B data

This section is for inspection only. The actual agent should continue to access data through tools.


In [ ]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

# Try common scaffold layouts.
candidate_dirs = [
    DATA_ROOT / "data_B",
    DATA_ROOT,
]

DATA_B = next((p for p in candidate_dirs if (p / "referrals.json").exists()), None)
if DATA_B is None:
    raise FileNotFoundError("Could not locate Problem B referrals.json")

referrals = load_json(DATA_B / "referrals.json")
patients = load_json(DATA_B / "patients.json")
specialties = load_json(DATA_B / "specialties.json")
urgency_bands = load_json(DATA_B / "urgency_bands.json")
clinic_slots = load_json(DATA_B / "clinic_slots.json")
contacts = load_json(DATA_B / "contacts.json")

# answer key may sit one level up
key_candidates = [
    DATA_ROOT / "expected_outcomes_B.json",
    DATA_B / "expected_outcomes_B.json",
]
ANSWER_KEY_PATH = next((p for p in key_candidates if p.exists()), None)
if ANSWER_KEY_PATH is None:
    raise FileNotFoundError("Could not locate expected_outcomes_B.json")

expected_outcomes = load_json(ANSWER_KEY_PATH)

print("Referrals       :", len(referrals))
print("Patients        :", len(patients))
print("Specialties     :", len(specialties))
print("Urgency bands   :", len(urgency_bands))
print("Clinic slots    :", len(clinic_slots))
print("Contacts        :", len(contacts))
print("Expected labels :", len(expected_outcomes))


In [ ]:
REFERRALS_BY_ID = {r["referral_id"]: r for r in referrals}
EXPECTED_BY_ID = {e["case_id"]: e for e in expected_outcomes}

print("First referral:")
pprint(referrals[0])

print("\nFirst expected label:")
pprint(expected_outcomes[0])


## 2.1 · Dataset integrity checklist

Before experiments:

```bash
python3 make_fixtures_B.py
python3 check_my_data.py
```

Expected final output:

```text
Your data hangs together.
```

Team-created cases must:
- use new IDs,
- preserve all shipped rows,
- be labelled before the agent is used to judge them,
- extend the same `expected_outcomes_B.json`.


# 3 · Dataset overview / EDA

**Owner: Liu + team**

Keep this lightweight. The point is to show evaluation-set composition, not to make decorative plots.

### TODO
- total cases
- supplied vs team-created
- expected-decision distribution
- case-family distribution
- negative-case count
- hostile/free-text count


In [ ]:
from collections import Counter

decision_counts = Counter(e["expected_decision"] for e in expected_outcomes)
family_counts = Counter(e.get("family", "unspecified") for e in expected_outcomes)

print("Expected decisions:", decision_counts)
print("Families:", family_counts)


# 4 · D0 — Why an agent?

**Owner: Team**

Complete this section once the main trajectories are working.

### What to explain

Problem B is not just one LLM call. The path varies according to tool observations.

Use at least three trajectories:

- `REF-5590` → red flag → early escalation
- `REF-5614` → missing mandatory test → request information
- `REF-5602` → full booking path

### Suggested conclusion structure

1. The next step is not fully known at the start.
2. Tool observations determine whether the run continues or stops.
3. The first irreversible action is `book_slot`.
4. A fixed workflow is possible for some substeps, but the agent is used to select among runtime branches and tool calls.
5. Agent flexibility costs extra turns/tokens, so D2/D3/D6 measure whether that flexibility is justified.

### Evidence to insert
- [ ] trajectory screenshots/table
- [ ] turns per representative case
- [ ] first-write governance point


# 5 · ROSHAN + SHEN — D1: Agent loop

**Primary owners: Roshan + Shen**

## What you are supposed to do

1. Understand `agent.py` block by block.
2. Verify the loop supports multiple calls in one turn.
3. Show that observations are fed back into the next turn.
4. Show that runs stop on a final decision or a guardrail.
5. Demonstrate at least one `book`, `request_information`, and `escalate` trajectory.

## What must remain in the modules

Do not rewrite the whole loop here.  
Changes to the real loop belong in `agent.py`.

## Mental model

```text
backend.next_move()
      ↓
one or more tool calls
      ↓
guardrail checks
      ↓
tools.call(...)
      ↓
observations
      ↓
transcript update
      ↓
next turn / final
```


In [ ]:
REPRESENTATIVE_CASES = ["REF-5602", "REF-5614", "REF-5590"]

for case_id in REPRESENTATIVE_CASES:
    print("\n" + "="*70)
    print(case_id)
    print("="*70)
    try:
        result = agent.run_case(case_id, problem="B", verbose=False)
        pprint(result)
    except Exception as exc:
        print("RUN FAILED:", exc)


### D1 inference placeholder

**Roshan/Shen add:**
- Why `REF-5602` needs more turns than the early-stop cases.
- Which calls are independent.
- Which calls are data-dependent.
- Why the final path is determined by observations rather than a fixed number of steps.


# 6 · ROSHAN + SHEN — D2(a): Tool design

Fill this from the actual `tools.py`.

| Tool | Purpose | Inputs | Returns | Read / Write | Why needed | Failure prevented |
|---|---|---|---|---|---|---|
| `get_referral` | | | | Read | | |
| `check_referral_criteria` | | | | Read/compute | | |
| `lookup_patient` | | | | Read | | |
| `get_clinic_slots` | | | | Read | | |
| `book_slot` | | | | **Write** | | |

### What Roshan/Shen must answer

For every tool:
1. Does the task fail without it?
2. Could the model confuse it with another tool?
3. Is the input restrictive enough to prevent invalid calls?
4. Is the returned observation bounded and relevant?
5. Could the tool be merged/split more cleanly?
6. Does the tool expose an irreversible action?

### Final D2(a) conclusion placeholder
Write 1–2 paragraphs defending the final tool set.


# 7 · ROSHAN + SHEN — D2(c): Sequential vs parallel calls

## Experiment question

Can we reduce model turns by grouping **only data-independent tool calls** while preserving the same underlying evidence, tool-call count, and final decision?

## Dependency rule

> Two calls may share a turn only if neither consumes the output of the other.

Examples:
- `check_referral_criteria` + `lookup_patient` → independent after referral fetch.
- two slot searches over disjoint windows → can be independent once band/window are known.
- `get_referral` → `check_referral_criteria` → dependent, so must remain sequential.


In [ ]:
# This experiment expects the shared scripted backend to already contain REF-5602.
# Keep the real implementation in backends.py. This cell is for measured comparison.

CASE_ID = "REF-5602"

parallel_result = agent.run_case(CASE_ID, problem="B", verbose=False)
print("PARALLEL/SHARED SCRIPT RESULT")
pprint(parallel_result)


### Sequential baseline

Roshan/Shen should add a temporary sequential script/override that performs the **same six calls** in separate turns:

```text
Turn 1  get_referral
Turn 2  check_referral_criteria
Turn 3  lookup_patient
Turn 4  get_clinic_slots #1
Turn 5  get_clinic_slots #2
Turn 6  book_slot
```

Then compare with the current grouped version:

```text
Turn 1  get_referral
Turn 2  check_referral_criteria + lookup_patient
Turn 3  get_clinic_slots #1 + get_clinic_slots #2
Turn 4  book_slot
```

### Required result table

| Metric | Sequential | Parallel |
|---|---:|---:|
| Tool calls | | |
| Turns | | |
| Scripted input-token estimate | | |
| Scripted output-token estimate | | |
| Final decision | | |
| Booking | | |
| Correctness preserved? | | |

### Required inference

Explain the reduction using:

\[
Input \approx B T + \frac{D T^2}{2}
\]

Clearly label scripted token numbers as **estimates**, not provider-measured usage.


# 8 · ROSHAN — Evaluation contribution (REF-6031–REF-6036)

**Owner: Roshan**

These cases test hostile/free-text safety and one benign control.

## What Roshan must add

For every case:
1. `EXTRA_*` data row(s)
2. expected label
3. case family
4. what the case tests
5. why the label is correct
6. why no earlier routing rule fires
7. `must_record`
8. confirmation `check_my_data.py` passes

## Assigned cases

| ID | Purpose | Expected family |
|---|---|---|
| REF-6031 | direct instruction injection | hostile free text |
| REF-6032 | fake `check_referral_criteria` output | hostile free text |
| REF-6033 | fake `book_slot` success output | hostile free text |
| REF-6034 | claimed authority / skip-checks instruction | hostile free text |
| REF-6035 | rephrased bypass instruction | hostile free text |
| REF-6036 | benign “book” wording control | ordinary / false-positive control |

### Important
The first five should be designed so the hostile instruction is the intended trigger.  
`REF-6036` should **not** escalate merely because it contains the word “book”.


## 8.1 · Reusable per-case template

Copy this block once per case:

```text
CASE ID:
OWNER:

CASE FAMILY:

WHAT THIS CASE TESTS:

DATA ADDED:
- EXTRA_REFERRALS:
- EXTRA_PATIENTS:
- EXTRA_CONTACTS:
- EXTRA_CLINIC_SLOTS:
- EXTRA_SPECIALTIES:

EXPECTED DECISION:

EXPECTED TRIGGER / MISSING / BOOKING:

WHY THIS IS THE CORRECT LABEL:

WHY NO EARLIER RULE FIRES:

MUST_RECORD:
1.
2.
3.

EXPECTED_OUTCOMES_B.JSON ENTRY:

VALIDATION:
[ ] make_fixtures_B.py ran
[ ] check_my_data.py → "Your data hangs together."
```


# 9 · SHEN SHUO — Personal contribution placeholder

**Owner: Shen Shuo**

## What Shen is supposed to do

### D1 / D2(a) / D2(c)
- Review Roshan's loop/tool changes.
- Validate the dependency rule for parallel calls.
- Add at least one additional edge trajectory to prove the shared loop is not over-fit to `REF-5602`.
- Verify tool observations are bounded and correct.
- Help maintain reusable scripted-family logic in `backends.py`.

### Evaluation cases
Assigned IDs: `REF-6021–REF-6026`

Focus:
- last legal day → book
- one day late → no-slot escalation
- capacity zero
- first full / second available
- multiple valid slots → earliest valid
- second boundary case in another specialty/band

### Live model
Run the frozen v2 system on **Claude 3 Haiku**.

Leave in notebook:
- trial count
- overall pass
- negative pass
- median turns
- input/output tokens
- cost
- notes on failures


# 10 · GONG XINYI — D2(b): Descriptor v1 → v2

**Owner: Gong Xinyi**

## What Gong is supposed to do

1. Create a deliberately weaker **v1** descriptor set.
2. Create improved **v2** descriptors using the agreed six-field structure.
3. Hold fixed:
   - model
   - evaluation cases
   - tool implementations
   - temperature/settings
4. Run both on the **same live model**.
5. Compare:
   - prompt tokens
   - trials
   - overall pass
   - negative pass
   - median turns
   - input/output tokens
   - cost
6. State which descriptor changes are retained and why.

> The scripted backend does not read the prompt, so scripted runs cannot prove v1→v2 prompt quality.


In [ ]:
descriptor_experiment = [
    {
        "version": "v1",
        "model": None,
        "trials": None,
        "prompt_tokens": None,
        "pass_rate": None,
        "negative_pass_rate": None,
        "median_turns": None,
        "input_tokens": None,
        "output_tokens": None,
        "cost_usd": None,
    },
    {
        "version": "v2",
        "model": None,
        "trials": None,
        "prompt_tokens": None,
        "pass_rate": None,
        "negative_pass_rate": None,
        "median_turns": None,
        "input_tokens": None,
        "output_tokens": None,
        "cost_usd": None,
    },
]
descriptor_experiment


# 11 · GONG XINYI — D3: Guardrails

**Owner: Gong Xinyi**

## What Gong is supposed to do

Run at least **10 guardrail cases** on the scripted backend.

Suggested coverage:

1. repeated identical action → de-duplication
2. step cap exceeded
3. token/budget ceiling exceeded
4. booking without confirmation
5. duplicate booking attempt
6. direct hostile instruction in referral free text
7. fake tool output in referral text
8. malformed/invalid tool call
9. write attempt before required evidence
10. benign text that should *not* trigger a hostile-text guard

At least 3 cases should involve hostile/free-text input.

### Required table

| Case | Expected guard | Fired? | Stopped by | Turns | Unsafe write count | Notes |
|---|---|---|---|---:|---:|---|


In [ ]:
guardrail_results = []

def add_guardrail_result(case_id, expected_guard, fired, stopped_by, turns, unsafe_writes, notes=""):
    guardrail_results.append({
        "case_id": case_id,
        "expected_guard": expected_guard,
        "fired": fired,
        "stopped_by": stopped_by,
        "turns": turns,
        "unsafe_writes": unsafe_writes,
        "notes": notes,
    })


# 12 · LIU XINYAO — D4: Evaluation harness

**Owner: Liu Xinyao**

## What Liu is supposed to do

1. Extend the shared harness so every labelled case can be scored.
2. Join results to `expected_outcomes_B.json` by `case_id`.
3. Keep **code checks** and **judgement checks** separate.

### Code-check examples
- final decision
- escalation trigger
- exact missing item
- booked clinic/date/time
- no unsafe booking where prohibited

### Judgement checks
Use human review or a clearly documented second-model judge for English `must_record` requirements.

If a model judges:
- commit the judge prompt,
- report the judge model,
- defend model-grading-model limitations.

### Trials
- ordinary cases: 1 trial/model
- negative cases: 3 trials/model

### Leave in notebook
- total cases
- negative cases
- hostile cases
- total trials/model
- pass-rate table
- negative-only pass-rate table


# 13 · LIU XINYAO — D5(a): Scripted reproducibility

**Owner: Liu Xinyao**

## Goal

A marker should clone the repo and run the final scripted system with:
- no API key
- no network
- no notebook clicking

Expected command:

```bash
python3 run_eval.py
```

Committed default:

```python
BACKEND = "scripted"
```

### What Liu should record
- number of cases with scripted coverage
- total scripted trials
- code-check pass count
- judgement-queue size/status
- median turns
- guardrail events
- results artefact path


# 14 · Run all referrals — shared scripted coverage

**Owner: Liu + Roshan/Shen**

This section is the team-wide integration check.

It attempts to run every referral for which the scripted backend has coverage.

Do not hide missing scripted coverage — list it explicitly.


In [ ]:
all_case_ids = [r["referral_id"] for r in referrals]

run_rows = []
missing_scripted = []

for case_id in all_case_ids:
    try:
        result = agent.run_case(case_id, problem="B", verbose=False)
        expected = EXPECTED_BY_ID.get(case_id, {})
        run_rows.append({
            "case_id": case_id,
            "expected": expected.get("expected_decision"),
            "actual": result.get("decision"),
            "turns": result.get("turns"),
            "tool_calls": len(result.get("evidence", [])),
            "backend": result.get("backend"),
            "stopped_by": result.get("stopped_by"),
        })
    except Exception as exc:
        missing_scripted.append((case_id, str(exc)))

print("Successfully run:", len(run_rows))
print("Missing/failed    :", len(missing_scripted))

if missing_scripted:
    print("\nCases needing scripted coverage or debugging:")
    for row in missing_scripted:
        print(row)


In [ ]:
# Lightweight display without pandas dependency.
for row in run_rows[:10]:
    print(row)


# 15 · All members — Evaluation-case tracker

| Member | IDs | Focus | Data added | Labels added | Checker passed |
|---|---|---|---|---|---|
| Gong | REF-6001–6006 | ordinary/run-length | ☐ | ☐ | ☐ |
| Liu | REF-6011–6016 | missing tests | ☐ | ☐ | ☐ |
| Shen | REF-6021–6026 | boundaries/slots | ☐ | ☐ | ☐ |
| Roshan | REF-6031–6036 | hostile/safety | ☐ | ☐ | ☐ |
| Xie | REF-6041–6046 | duplicates/history | ☐ | ☐ | ☐ |
| Zhong | REF-6051–6055 | mismatch/no-slot | ☐ | ☐ | ☐ |

### What every teammate must provide per case

1. case ID
2. `EXTRA_*` row(s)
3. family
4. purpose
5. expected decision
6. trigger / missing / booking
7. why label is correct
8. why no earlier rule fires
9. `must_record`
10. answer-key JSON row
11. checker confirmation

They do **not** need to write their own scripted backend implementation.


# 16 · Live model battery — shared section

**Owners: all members**

Only run this after the team freezes:
- v2 descriptors
- tools
- guardrails
- evaluation set
- answer key
- code commit/settings

## Experimental control

All final-v2 model runners use the same:
- 50-case set
- v2 prompt
- tools
- guardrails
- autonomy
- turn cap
- temperature/settings
- code commit

Only the model changes.

## Important

Live input/output tokens must come from the provider/API response.  
Do not report scripted estimates as measured usage.


In [ ]:
live_model_results = []

def add_live_result(
    owner, model, prompt_version, trials, passes,
    negative_trials, negative_passes,
    median_turns, input_tokens, output_tokens,
    cost_usd, run_date, notes=""
):
    live_model_results.append({
        "owner": owner,
        "model": model,
        "prompt_version": prompt_version,
        "trials": trials,
        "passes": passes,
        "pass_rate": passes / trials if trials else None,
        "negative_trials": negative_trials,
        "negative_passes": negative_passes,
        "negative_pass_rate": negative_passes / negative_trials if negative_trials else None,
        "median_turns": median_turns,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "cost_usd": cost_usd,
        "run_date": run_date,
        "notes": notes,
    })

# One row per member when measured.


### Model allocation

| Member | Run |
|---|---|
| Roshan | GPT-4o-mini v2 |
| Shen | Claude 3 Haiku v2 |
| Gong | Gemini 1.5 Flash v2 |
| Liu | Llama 3.1 8B v2 |
| Xie | Mistral Nemo v2 |
| Zhong | GPT-4o-mini **v1 pass** |


# 17 · XIE YULONG — D6: Cost-to-serve

**Owner: Xie Yulong**

Use **measured** model results from the live battery.

Problem B defaults:
- 4,000 referrals/month
- triage nurse = US$55/hour
- 10 minutes/failure
- fallback/failure cost ≈ US$9.17

## Minimum calculations

1. variable API/tool cost per referral
2. expected fallback cost:
   \[
   (1-p) \times failure\_cost
   \]
3. cost per successful referral
4. monthly cost at 4,000 referrals
5. ±10 percentage-point sensitivity around measured success
6. break-even success rate between cheap and expensive model
7. decision-driver discussion


In [ ]:
MONTHLY_VOLUME = 4000
NURSE_HOURLY_USD = 55
MINUTES_PER_FAILURE = 10
FAILURE_COST_USD = NURSE_HOURLY_USD * MINUTES_PER_FAILURE / 60

print("Default failure cost:", round(FAILURE_COST_USD, 2))


### Xie's required result table

| Model | Success rate | Variable cost/referral | Expected fallback | Cost/success | Monthly cost | Break-even note |
|---|---:|---:|---:|---:|---:|---|


# 18 · ZHONG YINGMEI — v1 pass + report/demo assembly

**Owner: Zhong Yingmei**

## v1 pass

Run GPT-4o-mini with the **v1 descriptor set** on the same evaluation set/settings used for the v2 comparison.

Leave:
- trial count
- overall pass
- negative pass
- turns
- tokens
- cost

## Report/demo assembly

Use committed result artefacts as the source of truth.

### Checklist
- architecture matches code
- D0 rationale uses real trajectories
- D2(a) tool choices justified
- D2(b) changes one variable
- D2(c) reports correctness + savings
- D3 table includes hostile input
- D4 reports trial counts
- D5 model table uses same frozen system
- D6 uses cost per successful task
- D7 has before/after evidence
- final recommendation is supported by measurements


# 19 · D7 — Failure reproduction #1

**Suggested layer: loop/control**

### Method
1. Start from working scripted agent.
2. Remove/disable action de-duplication.
3. Reproduce repeated identical tool calls.
4. Measure:
   - turns
   - cumulative input-token estimate
   - stopping reason
   - correctness
5. Restore dedupe and re-run.

### Required conclusion
Explain why pass/fail alone may miss this failure if the agent still reaches the correct final answer after wasting turns.


# 20 · D7 — Failure reproduction #2

**Suggested layer: tool/interface**

Recommended Problem B failure:
- weaken `get_clinic_slots` so it no longer constrains the urgency band / legal window,
- reproduce a wrong-slot or no-slot error,
- restore the correct interface.

### Required evidence
- working trace
- broken trace
- one deliberate change
- before/after metrics
- fix
- why this is a different failure mechanism from D7 #1


# 21 · Shared experiment registry

Every measured experiment should be appended here or mirrored in a committed result file.


In [ ]:
TEAM_RESULTS = []

def add_result(
    experiment_id, owner, question, change, fixed,
    backend, model=None, trials=None,
    pass_rate=None, negative_pass_rate=None,
    median_turns=None, input_tokens=None,
    output_tokens=None, cost_usd=None, notes=""
):
    TEAM_RESULTS.append({
        "experiment_id": experiment_id,
        "owner": owner,
        "question": question,
        "change": change,
        "fixed": fixed,
        "backend": backend,
        "model": model,
        "trials": trials,
        "pass_rate": pass_rate,
        "negative_pass_rate": negative_pass_rate,
        "median_turns": median_turns,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "cost_usd": cost_usd,
        "notes": notes,
    })


# 22 · Final comparison table

Populate only from measured/committed results.

| System/version | Backend/model | Trials | Overall pass | Negative pass | Median turns | Input tokens | Output tokens | Cost |
|---|---|---:|---:|---:|---:|---:|---:|---:|
| Scripted final | scripted | | | | | | | |
| Descriptor v1 | live / fixed model | | | | | | | |
| Descriptor v2 | live / same model | | | | | | | |
| Model 1 | live | | | | | | | |
| Model 2 | live | | | | | | | |
| Model 3 | live | | | | | | | |
| Model 4 | live | | | | | | | |
| Model 5 | live | | | | | | | |

## Final recommendation template

- Selected model:
- Selected prompt version:
- Autonomy:
- Turn cap:
- Overall pass:
- Negative pass:
- Cost per successful referral:
- Monthly cost @ 4,000:
- Break-even finding:
- Primary residual risk:
- Deployment recommendation:


# 23 · README / reproducibility checklist

Before submission:

### Fresh-clone test
- [ ] clone repo into a clean folder
- [ ] no API key
- [ ] no network required for scripted run
- [ ] `BACKEND = "scripted"` committed default
- [ ] `python3 run_eval.py` succeeds
- [ ] results reproduced from committed data/key
- [ ] README explains data path and commands

### Required artefacts
- [ ] edited `make_fixtures_B.py`
- [ ] generated Problem B JSON data
- [ ] extended `expected_outcomes_B.json`
- [ ] result tables / raw result JSON
- [ ] final agent/tool/guardrail/harness code
- [ ] D7 failure artefacts
- [ ] cost model
- [ ] `CONTRIBUTIONS.md`
- [ ] team declaration


# 24 · Final submission checklist

### Evaluation
- [ ] 50 total cases
- [ ] supplied rows unchanged
- [ ] every case has one answer-key label
- [ ] labels written independently of agent outputs
- [ ] `check_my_data.py` passes
- [ ] negative cases repeated as required
- [ ] at least 3 hostile/free-text cases
- [ ] code checks and judgement checks separated
- [ ] every reported pass rate includes trial count

### Live battery
- [ ] final v2 system frozen
- [ ] same eval set/tools/guards/settings for all final models
- [ ] multiple model families
- [ ] multiple price tiers
- [ ] actual provider usage captured

### Analysis
- [ ] D2(b) v1→v2 measured on same live model
- [ ] D2(c) sequential→parallel measured
- [ ] cost per success reported
- [ ] sensitivity reported
- [ ] break-even reported
- [ ] two deterministic failure reproductions

### Submission hygiene
- [ ] notebook outputs refreshed from final code
- [ ] no invented measurements
- [ ] README reproducible
- [ ] CONTRIBUTIONS.md matches commits
- [ ] every member can explain their section
